# TP4 — Lab 4: Stopword Removal

## Exercise 1 — Identifying stopwords

In [1]:
import nltk
from nltk.corpus import stopwords

stop_words = set(stopwords.words("english"))

text = "The student is working on a natural language processing project."
words = nltk.word_tokenize(text.lower())

print("stopwords found:", [word for word in words if word in stop_words])
print("content words:  ", [word for word in words if word not in stop_words])
print()
print("natural / language / processing:",
      {word: word in stop_words for word in ["natural", "language", "processing"]})
print()
print("first 15 stopwords:", stopwords.words("english")[:15])

stopwords found: ['the', 'is', 'on', 'a']
content words:   ['student', 'working', 'natural', 'language', 'processing', 'project', '.']

natural / language / processing: {'natural': False, 'language': False, 'processing': False}

first 15 stopwords: ['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't"]


### Answers — Exercise 1

1. **Stopwords in the example:** `the`, `is`, `on`, `a`.
2. **Complement:** `student`, `working`, `natural`, `language`, `processing`, `project`, `.` — a human can still guess the topic (an NLP project), so the content words carry the meaning.
3. **natural / language / processing:** none is in NLTK's list. That is expected: they are content words, not grammatical glue. Note that `processing` is only generic *inside* an NLP corpus (see Exercise 4).
4. **First 15 words:** `a, about, above, after, again, against, ain, all, am, an, and, any, are, aren, aren't`. Categories: articles/determiners (`a`, `an`), prepositions (`about`, `above`, `after`, `against`), adverbs/conjunctions (`again`, `all`, `any`), and forms of *be* / auxiliaries (`am`, `are`, `aren't`, `ain`).

**Questions**

1. The recognised stopwords are `the`, `is`, `on`, `a`.
2. No. `not` is very frequent but flips the meaning of a sentiment sentence; `is` can be the focus of a question-answering query.
3. Because removal helps when word identity matters more than grammar (search, topic classification) and hurts when small words carry meaning (sentiment, QA, translation).

## Exercise 2 — Quantifying the effect of stopword removal

In [2]:
import nltk
from nltk.corpus import stopwords

stop_words = set(stopwords.words("english"))

def tokenize(text):
    return nltk.word_tokenize(text.lower())

def stopword_stats(tokens, stop_words):
    remaining = [t for t in tokens if t not in stop_words]
    removed = len(tokens) - len(remaining)
    return {
        "original": len(tokens),
        "remaining": len(remaining),
        "removed": removed,
        "percent_removed": round(100 * removed / len(tokens), 1) if tokens else 0.0,
    }

text = "The student is working on a natural language processing project."
words = tokenize(text)
filtered = [w for w in words if w not in stop_words]

print("original:", words)
print("filtered:", filtered)
print()

samples = {
    "tweet": "I can't wait for this NLP lab, it is so cool!",
    "formal": "The committee has decided that the proposal will be reviewed during the next session.",
    "keywords": "NLP tokenization stopwords preprocessing pipeline",
}

print(f"{'style':10} {'original':>8} {'kept':>5} {'removed':>8} {'% removed':>10}")
for name, sentence in samples.items():
    stats = stopword_stats(tokenize(sentence), stop_words)
    print(f"{name:10} {stats['original']:>8} {stats['remaining']:>5} "
          f"{stats['removed']:>8} {stats['percent_removed']:>9}%")

original: ['the', 'student', 'is', 'working', 'on', 'a', 'natural', 'language', 'processing', 'project', '.']
filtered: ['student', 'working', 'natural', 'language', 'processing', 'project', '.']

style      original  kept  removed  % removed
tweet            14     8        6      42.9%
formal           15     7        8      53.3%
keywords          5     5        0       0.0%


### Answers — Exercise 2

1. **Original vs filtered (Exercise 1 sentence):** 11 tokens to 7; `the`, `is`, `on`, `a` are removed.
2. **`stopword_stats()`** returns original, remaining, removed and percentage removed (see the table).
3. **Table:** tweet 14 to 8 tokens (42.9% removed); formal 15 to 7 (53.3%); keywords 5 to 5 (0%).
4. **Stability:** the percentage varies a lot (0–53%). A keyword list has almost no function words, formal prose is dense in them, and a tweet sits in between. So the effect depends on style, not just length.

## Exercise 3 — The importance of context: negation

In [3]:
import nltk
from nltk.corpus import stopwords

stop_words = set(stopwords.words("english"))

sentences = [
    "This movie is not good.",
    "This movie is really not that bad, actually.",
]

positive = {"good", "great", "excellent"}
negative = {"bad", "terrible", "awful"}
negators = {"not", "no", "never", "n't", "nor"}

def tokens(text):
    return nltk.word_tokenize(text.lower())

def remove_stopwords(toks, sw=stop_words):
    return [t for t in toks if t not in sw]

def score(toks, handle_negation=False):
    total = 0
    for i, tok in enumerate(toks):
        value = 1 if tok in positive else -1 if tok in negative else 0
        if value and handle_negation and any(w in negators for w in toks[max(0, i - 2):i]):
            value = -value
        total += value
    return total

print("WITHOUT stopword removal")
for s in sentences:
    toks = tokens(s)
    print(f"  {s}  tokens={toks}  score={score(toks)}")

print("\nWITH stopword removal (NLTK list)")
for s in sentences:
    toks = remove_stopwords(tokens(s))
    print(f"  {s}  tokens={toks}  score={score(toks)}")

fixed_stop_words = stop_words - negators
print("\nWith negators kept + negation-aware scorer")
for s in sentences:
    toks = [t for t in tokens(s) if t not in fixed_stop_words]
    print(f"  {s}  tokens={toks}  score={score(toks, handle_negation=True)}")

WITHOUT stopword removal
  This movie is not good.  tokens=['this', 'movie', 'is', 'not', 'good', '.']  score=1
  This movie is really not that bad, actually.  tokens=['this', 'movie', 'is', 'really', 'not', 'that', 'bad', ',', 'actually', '.']  score=-1

WITH stopword removal (NLTK list)
  This movie is not good.  tokens=['movie', 'good', '.']  score=1
  This movie is really not that bad, actually.  tokens=['movie', 'really', 'bad', ',', 'actually', '.']  score=-1

With negators kept + negation-aware scorer
  This movie is not good.  tokens=['movie', 'not', 'good', '.']  score=-1
  This movie is really not that bad, actually.  tokens=['movie', 'really', 'not', 'bad', ',', 'actually', '.']  score=1


### Answers — Exercise 3

1. **Removed negation:** `not` in both sentences. In the second, `that` is also a stopword, but it is not the negator.
2. **Naive scores:** sentence 1 = +1, sentence 2 = -1, both before and after removal. Removing `not` changes nothing because the scorer never looked at it — that is the real weakness.
3. **Fix:** keep `not/no/never/n't/nor` and flip the polarity of a sentiment word preceded by a negator within two tokens. Scores become -1 ("not good") and +1 ("not ... bad"), which matches the meaning.

**Questions**

1. The negation disappears: "not good" turns into "good".
2. `not` was removed by the NLTK list; the naive scorer ignored it before and after, so the score was unchanged. With the fixed list and negation rule it is kept and flips the score.
3. Sentiment often depends on small words (`not`, `no`, `never`, `very`, `really`); blanket removal can invert or weaken the polarity.

## Exercise 4 — Custom, domain-specific stopword lists

In [4]:
import nltk
from collections import Counter
from nltk.corpus import stopwords

stop_words = set(stopwords.words("english"))
custom_stopwords = {"the", "is", "a", "on"}

print("NLTK only:", len(stop_words - custom_stopwords))
print("in NLTK but not custom (sample):", sorted(stop_words - custom_stopwords)[:20])
print("in custom but not NLTK:", custom_stopwords - stop_words)

corpus = [
    "Natural language processing helps computers understand text.",
    "This course covers tokenization and text processing.",
    "NLP pipelines often start with normalization.",
    "Language models are trained on large text collections.",
    "Processing speech is different from processing written language.",
]

corpus_tokens = [nltk.word_tokenize(s.lower()) for s in corpus]
flat = [t for toks in corpus_tokens for t in toks if t.isalpha()]
freq = Counter(flat)
print("\nraw frequencies:", freq.most_common())

domain_stopwords = {"processing", "language", "text"}

def vocabulary(stop):
    return {t for t in flat if t not in stop}

nltk_vocab = vocabulary(stop_words)
extended_vocab = vocabulary(stop_words | domain_stopwords)
print("\nNLTK only   :", len(nltk_vocab), sorted(nltk_vocab))
print("NLTK + domain:", len(extended_vocab), sorted(extended_vocab))
print("removed by domain list:", sorted(nltk_vocab - extended_vocab))

NLTK only: 194
in NLTK but not custom (sample): ['about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been', 'before']
in custom but not NLTK: set()

raw frequencies: [('processing', 4), ('language', 3), ('text', 3), ('natural', 1), ('helps', 1), ('computers', 1), ('understand', 1), ('this', 1), ('course', 1), ('covers', 1), ('tokenization', 1), ('and', 1), ('nlp', 1), ('pipelines', 1), ('often', 1), ('start', 1), ('with', 1), ('normalization', 1), ('models', 1), ('are', 1), ('trained', 1), ('on', 1), ('large', 1), ('collections', 1), ('speech', 1), ('is', 1), ('different', 1), ('from', 1), ('written', 1)]

NLTK only   : 22 ['collections', 'computers', 'course', 'covers', 'different', 'helps', 'language', 'large', 'models', 'natural', 'nlp', 'normalization', 'often', 'pipelines', 'processing', 'speech', 'start', 'text', 'tokenization', 'trained', 'understand', 'written']
NLTK + domain: 19 ['collect

### Answers — Exercise 4

1. **Custom vs NLTK:** NLTK has 198 words, the custom list 4. 194 words are in NLTK but not in the custom list; the custom list is a subset, so nothing is in custom but not NLTK.
2. **Frequencies:** `processing` 4, `language` 3, `text` 3 (punctuation ignored); the rest appear once.
3. **Extension:** add `processing`, `language` and `text` — frequent across the corpus but weak at telling documents apart, especially in an NLP course.
4. **Vocabulary:** NLTK only leaves 22 distinct words; NLTK + domain leaves 19. The three added words are exactly what disappears.

**Question**

A generic list cannot know that `processing` is noise in an NLP corpus, or that a word like `not` is critical for sentiment. A domain list adapts the removal to the corpus and the task.

## Exercise 5 — Integrated preprocessing mini-pipeline

In [5]:
import re
import nltk
from nltk.corpus import stopwords

text = ("Natural Language Processing (NLP) has grown VERY fast since 2018!! "
        "Companies such as Google and OpenAI now offer APIs at "
        "https://api.example.com, and models cost around $0.002 per 1,000 tokens. "
        "Isn't that amazing?")

url_pattern = r"https?://[^\s,()]*[^\s,().]"
price_pattern = r"\$\d+(?:,\d{3})*(?:\.\d+)?"
number_pattern = r"\d+(?:,\d{3})*(?:\.\d+)?"

# Stage 1: mask the URL and the price
masked = re.sub(url_pattern, "URLTOKEN", text)
masked = re.sub(price_pattern, "PRICETOKEN", masked)
regex_view = masked.replace("URLTOKEN", "<URL>").replace("PRICETOKEN", "<PRICE>")
print("1) after RegEx:\n  ", regex_view)

# Stage 2: tokenize
tokens = nltk.word_tokenize(masked)
print("\n2) after tokenization:\n  ",
      [t.replace("URLTOKEN", "<URL>").replace("PRICETOKEN", "<PRICE>") for t in tokens])

# Stage 3: lowercase and mask numbers
normalized = []
for tok in tokens:
    low = tok.lower()
    if low == "urltoken":
        normalized.append("<URL>")
    elif low == "pricetoken":
        normalized.append("<PRICE>")
    elif re.fullmatch(number_pattern, tok):
        normalized.append("<NUM>")
    else:
        normalized.append(low)
print("\n3) after normalization:\n  ", normalized)

# Stage 4: remove stopwords, but keep negation
negators = {"not", "no", "never", "n't", "nor"}
sw = set(stopwords.words("english")) - negators
filtered = [t for t in normalized if t not in sw]
print("\n4) after stopword removal (negation kept):\n  ", filtered)

# Stage 5: token count after every stage
raw_tokens = nltk.word_tokenize(text)
stages = [
    ("raw", raw_tokens),
    ("after RegEx", nltk.word_tokenize(masked)),
    ("after tokenization", tokens),
    ("after normalization", normalized),
    ("after stopword removal", filtered),
]
print(f"\n{'stage':25}{'tokens':>7}")
for name, toks in stages:
    print(f"{name:25}{len(toks):>7}")

1) after RegEx:
   Natural Language Processing (NLP) has grown VERY fast since 2018!! Companies such as Google and OpenAI now offer APIs at <URL>, and models cost around <PRICE> per 1,000 tokens. Isn't that amazing?

2) after tokenization:
   ['Natural', 'Language', 'Processing', '(', 'NLP', ')', 'has', 'grown', 'VERY', 'fast', 'since', '2018', '!', '!', 'Companies', 'such', 'as', 'Google', 'and', 'OpenAI', 'now', 'offer', 'APIs', 'at', '<URL>', ',', 'and', 'models', 'cost', 'around', '<PRICE>', 'per', '1,000', 'tokens', '.', 'Is', "n't", 'that', 'amazing', '?']

3) after normalization:
   ['natural', 'language', 'processing', '(', 'nlp', ')', 'has', 'grown', 'very', 'fast', 'since', '<NUM>', '!', '!', 'companies', 'such', 'as', 'google', 'and', 'openai', 'now', 'offer', 'apis', 'at', '<URL>', ',', 'and', 'models', 'cost', 'around', '<PRICE>', 'per', '<NUM>', 'tokens', '.', 'is', "n't", 'that', 'amazing', '?']

4) after stopword removal (negation kept):
   ['natural', 'language', 'proc

### Answers — Exercise 5

- **RegEx:** URL → `<URL>`, price → `<PRICE>`.
- **Normalization:** lowercase, and `2018` / `1,000` → `<NUM>`.
- **Negation decision:** keep `n't` (and its relatives) instead of removing it. "Isn't that amazing?" is a positive rhetorical question, and Exercise 3 showed that dropping negation can flip sentiment. NLTK does not list `n't`, but `is` is removed, so the negation cue survives on purpose.
- **Token counts:** 43 raw → 40 after RegEx → 40 after tokenization → 40 after normalization → 30 after stopword removal. The raw count is higher because the URL is split into several tokens before masking.